In [20]:
import pandas as pd 
import numpy as np

In [21]:
df=pd.read_csv("C:\\Users\\Jathin Prakash\\Desktop\\final_dataset.csv")

In [22]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [23]:
movies = df[["title", "tags"]]

movies.head()

,title,tags
0,Toy Story,Animation Comedy Family jealousy toy boy frien...
1,Jumanji,Adventure Fantasy Family boardgame disappearan...
2,Grumpier Old Men,Romance Comedy fishing bestfriend duringcredit...
3,Father of the Bride Part II,Comedy baby midlifecrisis confidence aging dau...
4,Heat,Action Crime Drama Thriller robbery detective ...


In [24]:
movies=movies.dropna()

In [25]:
len(movies)

9184

In [26]:
from nltk.stem.porter import PorterStemmer
import nltk

# Download once
nltk.download('punkt')

ps = PorterStemmer()

# Apply stemming
movies['tags'] = movies['tags'].apply(
    lambda x: " ".join([ps.stem(word) for word in x.split()])
)

[nltk_data] Downloading package punkt to C:\Users\Jathin
[nltk_data]     Prakash\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [27]:
tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=15000,
    ngram_range=(1,2)
)

vectors = tfidf.fit_transform(movies['tags'])

print(vectors.shape)

(9184, 15000)


In [28]:
similarity = cosine_similarity(vectors)

print(similarity.shape)

(9184, 9184)


In [29]:
def recommend(movie_name, top_n=5):
    
    # Check whether movie exists
    if movie_name not in movies['title'].values:
        print("Movie not found in dataset")
        return

    # Get movie index
    movie_index = movies[movies['title'] == movie_name].index[0]

    # Get similarity scores
    distances = similarity[movie_index]

    # Sort movies based on similarity
    movie_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:top_n+1]
    print(f"\nRecommended movies for '{movie_name}':\n")

    for i, movie in enumerate(movie_list, start=1):
        index = movie[0]
        score = movie[1]

        print(f"{i}. {movies.iloc[index].title}  | Similarity Score: {score:.4f}")

In [30]:
recommend("Interstellar")


Recommended movies for 'Interstellar':

1. Apollo 13  | Similarity Score: 0.2591
2. Love  | Similarity Score: 0.2284
3. The Martian  | Similarity Score: 0.2190
4. Silent Running  | Similarity Score: 0.1899
5. Passengers  | Similarity Score: 0.1844


In [31]:
import pickle

pickle.dump(movies, open('movies.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))

In [32]:
recommend("Heat")


Recommended movies for 'Heat':

1. No Good Deed  | Similarity Score: 0.2847
2. Setup  | Similarity Score: 0.1882
3. Armored  | Similarity Score: 0.1870
4. Gone Baby Gone  | Similarity Score: 0.1703
5. Kiss Kiss Bang Bang  | Similarity Score: 0.1686
